## Загрузка датасета GorillaHard на HuggingFace

Ноутбук собирает `test.json` и `shots.json` из `datasets/GorillaHard/` в `datasets.DatasetDict`
и заливает его на 🤗 Hub по указанному пути.

Что здесь специфично для GorillaHard:
* поле `instruction` в локальных файлах — это **индекс** промпта в `dataset_meta.json["prompts"]`;
  перед заливкой он заменяется на сам текст промпта (общее правило MERA);
* `inputs.context` и `inputs.tools` — сырой файл и JSON-каталог, оба полны фигурных скобок,
  поэтому промпт собирается заменой подстрок, а не `str.format`;
* `outputs` — это и есть правильный ответ (конверт вызова или отказа), по нему считаются
  содержательные метрики. Про режим приватной заливки читайте предупреждение ниже;
* перед заливкой прогоняется проверка целостности: каждый эталонный ответ должен получать
  `sample_pass_rate = 1.0` у того же скорера, которым считаются метрики.

In [ ]:
import json
import os
import sys

import datasets
from tqdm import tqdm

### Подготовка данных

#### WARNING!

Если ваш датасет является __ПРИВАТНЫМ__, оставьте `MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS` равным `True`.
Иначе поставьте `False`. Этот флаг дальше используется, чтобы стереть ответы перед загрузкой на ХФ.
На ХФ даже приватно не должно лежать датасетов с ответами!

⚠️ Для GorillaHard `outputs` — это сам правильный ответ, а не свидетельство решаемости: по нему
сверяются вид конверта, инструмент и аргументы. Если стереть ответы, то по копии с Hub можно
посчитать только `format_pass_rate` и `constraint_pass_rate` — остальные метрики скорер осознанно
не выдаёт (см. `benchmark_tasks/gorillahard/utils.py`), чтобы отсутствие эталона не выглядело как
провал модели. Полную оценку в этом случае проводит организатор бенчмарка на непубличной копии.

In [ ]:
MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS = True

Параметр `path_to_data` — путь ДО файлов `shots.json` и `test.json`.

Параметр `path_to_meta` — путь ДО `dataset_meta.json`.

Пути ниже указаны относительно расположения ноутбука в `utils/`; поменяйте, если запускаете из другого места.

In [ ]:
path_to_data = "../datasets/GorillaHard/"
path_to_meta = "../datasets/GorillaHard/"
path_to_task = "../benchmark_tasks/gorillahard/"

Сплиты и мета лежат в формате JSON.

In [ ]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

#### Подгрузка данных

In [ ]:
shots = load_json(os.path.join(path_to_data, "shots.json"))["data"]
test = load_json(os.path.join(path_to_data, "test.json"))["data"]
meta = load_json(os.path.join(path_to_meta, "dataset_meta.json"))

print(f"shots: {len(shots)}, test: {len(test)}")

Из меты для датасета нужны только промпты.

In [ ]:
prompts = meta["prompts"]
len(prompts)

#### Обработка полей датасета

На ХФ загружается датасет, где у КАЖДОГО сэмпла вместо числа в поле `instruction` стоит промпт.
Число указывает, какой по индексу взять промпт из секции с промптами в мете датасета.

Ячейка идемпотентна: если её случайно выполнить дважды, строки не будут перезаписаны повторно.

In [ ]:
def resolve_prompts(split):
    for card in split:
        if isinstance(card["instruction"], int):
            card["instruction"] = prompts[card["instruction"]]


resolve_prompts(shots)
resolve_prompts(test)

print(test[0]["instruction"][:200], "...")

#### Проверка целостности перед заливкой

Собираем промпт ровно так, как это будет делать lm-eval (`utils.doc_to_text` — замена подстрок,
не `str.format`), и прогоняем эталонный ответ через тот же скорер, которым считается метрика.
Если что-то не сходится — на Hub такой датасет заливать нельзя.

In [ ]:
sys.path.insert(0, os.path.abspath(path_to_task))
import utils as U

bad_prompt, bad_gold = [], []
for card in tqdm(shots + test):
    rendered = U.doc_to_text(card)
    if not rendered.strip() or "{question}" in rendered or "{tools}" in rendered:
        bad_prompt.append(card["meta"]["id"])
    res = U.process_results(card, [card["outputs"]])
    if res.get("sample_pass_rate") != 1.0:
        failed = [c["name"] for c in U.score_response(card, card["outputs"])["format_checks"]
                  if not c["pass"]]
        bad_gold.append((card["meta"]["id"], failed))

print("промптов, которые не собираются:", len(bad_prompt))
print("эталонных ответов, не проходящих проверку:", len(bad_gold))
assert not bad_prompt and not bad_gold, (bad_prompt[:5], bad_gold[:5])

#### Убираем ответы для приватных задач

Надеемся, вы поставили в начале ноутбука корректное значение `MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS`.

Если там стоит `True`, то в `test` сплите ответы на все задания стираются. Вместо них остаётся пустая
строка, чтобы вы случайно не пушнули на ХФ датасет с заполненными ответами, и они не утекли.
Ответы в `shots` сохраняются: это few-shot примеры, они и должны быть видны модели.

In [ ]:
def hide_answers(dataset_split: list):
    for card in tqdm(dataset_split):
        card["outputs"] = ""

In [ ]:
if MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS:
    hide_answers(test)

### Создаем датасет для загрузки на ХФ

#### Аннотация полей датасета

В `features` повторяется структура КАЖДОГО сэмпла датасета с описанием формата данных в каждом поле.

Два места, специфичных для GorillaHard:
* `inputs.tools` — **строка** с JSON-каталогом, а не вложенная структура: у инструментов разное
  число параметров, фиксированной схемой их не описать, и модель всё равно видит каталог текстом;
* `inputs.context` — строка, у 158 вопросов пустая. Поле должно присутствовать всегда, иначе схема
  разъедется.

In [ ]:
features = datasets.Features({
    "instruction": datasets.Value("string"),
    "inputs": {
        "question": datasets.Value("string"),
        "context": datasets.Value("string"),
        "tools": datasets.Value("string"),
        "format": datasets.Value("string"),
    },
    "outputs": datasets.Value("string"),
    "meta": {
        "id": datasets.Value("int32"),
        "base_id": datasets.Value("string"),
        "categories": {
            "language": datasets.Value("string"),
            "difficulty": datasets.Value("string"),
            "family": datasets.Value("string"),
            "answer_kind": datasets.Value("string"),
            "n_tools": datasets.Value("int32"),
            "has_context": datasets.Value("string"),
            "corpus_format": datasets.Value("string"),
        },
        "annotation": {
            "is_solvable": datasets.Value("string"),
            "language_correctness": datasets.Value("string"),
            "answer_correctness": datasets.Value("string"),
            "positive_votes": datasets.Value("int32"),
        },
    },
})

#### Создание датасетов для каждого сплита

Сэмплы тяжёлые — в `inputs.context` лежит целый файл, — поэтому тестовый сплит собирается кусками
по `STEP` сэмплов: так конвертация идёт заметно быстрее, чем одним вызовом на все 500.

In [ ]:
shots_ds = datasets.Dataset.from_list(shots, features=features)

STEP = 50
chunks = []
for i in tqdm(range(0, len(test), STEP)):
    chunks.append(datasets.Dataset.from_list(test[i: i + STEP], features=features))
test_ds = datasets.concatenate_datasets(chunks)

shots_ds, test_ds

##### Проверка

Проверим, что сборка прошла успешно — ничего не потеряно, не продублировано и не переехало.

In [ ]:
# количество вопросов до конвертации и после совпадает
assert len(test) == len(test_ds) and len(shots) == len(shots_ds)

# id вопросов сходятся и остаются сквозными: 1..5 в shots, 6..505 в test
assert [c["meta"]["id"] for c in test] == [c["meta"]["id"] for c in test_ds]
assert [c["meta"]["id"] for c in shots] == [c["meta"]["id"] for c in shots_ds]
ids = sorted(c["meta"]["id"] for c in shots + test)
assert ids == list(range(1, len(ids) + 1))

# каталог инструментов пережил конвертацию и всё ещё парсится
assert all(json.loads(c["inputs"]["tools"]) for c in test_ds)
print("OK")

#### Собираем сплиты в один датасет

In [ ]:
dataset = datasets.DatasetDict({"shots": shots_ds, "test": test_ds})
dataset

### Загрузка датасета на ХФ

Для загрузки на ХФ понадобятся:
- Токен — строка с ключом, дающим право записи в репозиторий.
- Путь для записи — аккаунт и название датасета. Название пишите ровно так, как оно заявлено в мете
  (`dataset_meta.json["dataset_name"]`), регистр имеет значение.

Советуем сначала залить всё приватно и выслать на почту mera@a-ai.ru токен и путь для верификации.

In [ ]:
from dotenv import load_dotenv

# Загружаем переменные из .env файла
load_dotenv('../.env')

### TOKEN
token = os.getenv('HF_TOKEN')
if token is None:
    raise ValueError("HF_TOKEN not found in .env file")

### UPLOAD PATH — поменяйте на нужный вам путь
dataset_path_hub = "MERA-evaluation/GorillaHard"

# Чтобы предварительно посмотреть, как датасет будет выглядеть после заливки,
# можно сначала загрузить его в свой приватный репозиторий:
# dataset_path_hub = "<your-account>/GorillaHard"

### PRIVATE OR PUBLIC
upload_private = True

print("upload to:", dataset_path_hub, "| private:", upload_private)

In [ ]:
dataset.push_to_hub(dataset_path_hub, private=upload_private, token=token)

### Проверка того, как датасет загрузился на ХФ

Загрузим датасет обратно и убедимся, что его увидит корректно любой, кто его скачает:
все поля на месте, содержание совпадает с исходным, а промпт по-прежнему собирается.

In [ ]:
ds = datasets.load_dataset(dataset_path_hub, token=token)
ds

In [ ]:
check = []
for idx, card in enumerate(ds["test"]):
    same_question = test[idx]["inputs"]["question"] == card["inputs"]["question"]
    same_tools = test[idx]["inputs"]["tools"] == card["inputs"]["tools"]
    check.append(same_question and same_tools)

all(check)

Финальная проверка: промпт собирается из скачанной с Hub копии — ровно то, что будет делать lm-eval.
После этого задачу можно запускать по HF-конфигу:

```bash
lm_eval --tasks gorillahard --include_path ./benchmark_tasks \
        --model hf --model_args pretrained=<model> --output_path ./out --log_samples
```

In [ ]:
card = ds["test"][0]
print(U.doc_to_text(card)[:600], "...")